# About this notebook.

This notebook goes through all the texts and aplies on them a NLP pipeline consisting of (1) cleaning of the raw text, (2) sentence tokenization, (3) part-of-speech annotation, (4) lemmatization, and (5) named entity recognition.

The processed textual data are saved for future reuse.

In [1]:
import spacy
import os
import glob
from spacy.tokens import Doc
from spacy.language import Language
import pickle
from unidecode import unidecode
import sddk
import pandas as pd
import re
import sys
import importlib
import json
from spacy.tokens import Token
from spacy.language import Language
import google_conf
import pandas as pd

/home/jupyter-vojta/notebooks/labyrinth/venv_torch_nlp/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
emlap_metadata = pd.read_csv("../data/emlap_metadata.csv", sep=";")
emlap_metadata.head(5)

,working_title,filenames,no.,is_done,is_noscemus,if_noscemus_id,AUTHORSHIP,is_one_author,#if more than 1 author skip section and choose compendium below,is_author_known,...,publisher_comments,CONTENTS,genre,subject,SOURCE OF FILE,link,source_of_file,origin_of_copy,other_notes,tokens_N
0,"Augurello, Chrysopoeia",100001_Augurello1515_Chrysopoeia_GB_Noscemus,100001,True,True,713324.0,NaN,True,NaN,True,...,NaN,NaN,didactic poem,alchemy,NaN,https://wiki.uibk.ac.at/noscemus/Chrysopoeia,GB,Noscemus,NaN,23718
1,"Pseudo-Lull, Secretis",100002_Pseudo-Lull1518_De secretis_naturae_MDZ...,100002,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,treatise,"alchemy, medicine",NaN,https://www.digitale-sammlungen.de/en/view/bsb...,MDZ,MBS,NaN,24673
2,"Pantheus, Ars Transmutatione",100003_Pantheus1518_Ars_Transmutationis_Metall...,100003,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,treatise,alchemy,NaN,https://www.google.co.uk/books/edition/Ars_Tra...,GB,BL,NaN,8646
3,"Anon, Vera alchemiae",100004_Anon1561_Verae_Alchemiae_MDZ_MBS,100004,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,"compendium, florilegium",alchemy,NaN,https://mdz-nbn-resolving.de/details:bsb10141168,MDZ,MBS,NaN,3521
4,"Pantheus, Voarchadumia",100005_Pantheus1530_Voarchadumia_ONB,100005,True,False,NaN,NaN,True,NaN,True,...,NaN,NaN,treatise,alchemy,NaN,https://data.onb.ac.at/rep/10588E49,ONB,ONB,NaN,20386


In [3]:
len(emlap_metadata)

100

In [4]:
#filename_id_dict = dict(zip(emlap_metadata["filename"], emlap_metadata["No."]))

For preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level one level up.

The module can be clonned from here: https://github.com/CCS-ZCU/latin-preprocessing and imported to python following the steps below:

In [5]:
# for preprocessing the latin texts, we will use a module located outside of the current repository, specifically at the same level as the current project.
current_working_directory = os.getcwd()
relative_path = '../../latin-preprocessing/' # change according to your location...
module_path = os.path.abspath(os.path.join(current_working_directory, relative_path))
if module_path not in sys.path:
    sys.path.insert(0, module_path)
# Now import the module
import tomela

Using CPU for spaCy.


Tomela contains tuned latin preprocessing pipeline relying on spaCy and latinCy. You can check the pipeline as here:

In [6]:
tomela.nlp.pipeline

[('senter', <spacy.pipeline.senter.SentenceRecognizer at 0x7bb466495370>),
 ('normer', <function la_core_web_lg.functions.normer(doc)>),
 ('tok2vec', <spacy.pipeline.tok2vec.Tok2Vec at 0x7bb466495730>),
 ('tagger', <spacy.pipeline.tagger.Tagger at 0x7bb4664949b0>),
 ('morphologizer',
  <spacy.pipeline.morphologizer.Morphologizer at 0x7bb5453cfef0>),
 ('trainable_lemmatizer',
  <spacy.pipeline.edit_tree_lemmatizer.EditTreeLemmatizer at 0x7bb466494fb0>),
 ('parser', <spacy.pipeline.dep_parser.DependencyParser at 0x7bb4669205f0>),
 ('lookup_lemmatizer',
  <function la_core_web_lg.functions.make_lookup_lemmatizer_function(doc)>),
 ('ner', <spacy.pipeline.ner.EntityRecognizer at 0x7bb466920660>),
 ('remorpher', <function la_core_web_lg.functions.remorpher(doc)>)]

In [7]:
tomela.nlp.max_length = 4000000

In [8]:
doc = tomela.nlp("Veritas, vt vlla dicit, semper est universalis et a principiis fundamentalis oritur (lib. 3, cap. VI)")
for token in doc:
    print((token.text, token.lemma_, token.pos_))

('Veritas', 'ueritas', 'NOUN')
(',', ',', 'PUNCT')
('vt', 'vt', 'ADV')
('vlla', 'vllus', 'NOUN')
('dicit', 'dico', 'VERB')
(',', ',', 'PUNCT')
('semper', 'semper', 'ADV')
('est', 'sum', 'AUX')
('universalis', 'uniuersalis', 'ADJ')
('et', 'et', 'CCONJ')
('a', 'ab', 'ADP')
('principiis', 'principium', 'NOUN')
('fundamentalis', 'fundamentalis', 'ADJ')
('oritur', 'orior', 'VERB')
('(lib', '(lib', 'NOUN')
('.', '.', 'PUNCT')
('3', '3', 'NUM')
(',', ',', 'PUNCT')
('cap', 'capitulum', 'NOUN')
('.', '.', 'PUNCT')
('VI', 'uis', 'NUM')
(')', ')', 'PUNCT')


In [9]:
source_path = "/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/"
len(os.listdir(source_path))

200

In [10]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/EMLAP_2025-10-31/annotated_textblocks/", "../data/emlap_annotated_textblocks/", dirs_exist_ok=True)

'../data/emlap_annotated_textblocks/'

In [11]:
[f for f in sorted(os.listdir(source_path)) if "_params" not in f]

['100001_Augurello1515_Chrysopoeia_GB_Noscemus.json',
 '100002_Pseudo-Lull1518_De_secretis_naturae_MDZ_MBS.json',
 '100003_Pantheus1518_Ars_Transmutationis_Metallicae_BL_GB.json',
 '100004_Anon1561_Verae_Alchemiae_MDZ_MBS.json',
 '100005_Pantheus1530_Voarchadumia_ONB.json',
 '100006_Savonarola1532_De_arte_conficiendi_aquam_vitae_ONB.json',
 '100007_Anon1550_Rosarium_philosophorum_ER_ZZ.json',
 '100008_Severinus1572_Epistola_MBZ_MBS.json',
 '100009_Vegius1518_Inter_inferiora_corpora_disputatio_ONB.json',
 '100010_Bracesco1548_De_alchemia_dialogi_duo_IA_Madrid.json',
 '100011_Anon1541_De_alchemia_MDZ_MBS.json',
 '100012_Gessner1552_Thesaurus_Euonymi_Philiatri_ER_ZZ.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100014_Toxites1567_Spongia_stibii_MDZ_MBS.json',
 '100015_Gessner1569_Thesaurus_Euonymi_Philiatri_liber_secundus_MDZ_MBS.json',
 '100016_Bonus1546_Pretiosa_Margarita_Novella_ONB.json',
 '100017_Bodenstein1559_Isagoge_MDZ_MBS.json',
 '100018_Trevisanus1567_Pe

In [12]:
files_overview = []
for filename in os.listdir(source_path):
    #filename = 'DuChesne1575_Ad_Iacobi_Auberti_MDZ_Augsburg.json'
    if "_params" not in filename:
        filepath = os.path.join(source_path, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            textblocks = json.load(f)
        pages_n = len(textblocks)
        chars_n = sum([sum([len(tb["text"]) for tb in p]) for p in textblocks])
        files_overview.append({"filename" : filename, "pages_n" : pages_n, "chars_n" : chars_n})
files_processed = pd.DataFrame(files_overview)
files_processed

,filename,pages_n,chars_n
0,100084_Croll1609_Basilica_chymica_MDZ_MBS.json,477,694793
1,100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json,508,694493
2,100058_Hagecius1596_Actio_medica_ER_UBB.json,89,104677
3,100013_Ulstad1525_Coelum_philosophorum_Medica_...,113,237793
4,100069_Severinus1571_Idea_medicinae_philosophi...,463,600046
...,...,...,...
95,100056_Claveus1598_Apologia_crysopoeiae_MDZ_MB...,233,233328
96,100065_Libavius1594_Neoparacelsica_MDZ_MBS.json,821,1079977
97,100031_Phaedro1562_Aquila_coelestis_MBZ_MBS.json,55,15562
98,100098_Burggravius1630_Biolychnium_VD17_SLUB.json,167,186267


In [11]:
#emlap_catalogue = google_conf.setup(sheet_url="https://docs.google.com/spreadsheets/d/1bkHHTYc86K2IuEXqfYfkDNt5LovtvCU3gvqHIbVio88/edit?usp=sharing", service_account_path="../../../ServiceAccountsKey.json")

# google_conf.set_with_dataframe(emlap_catalogue.add_worksheet("files_processed", 1,1), files_processed, include_index=False)


# Develop and test with one example test

In [13]:
filename = '100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json'
filepath = os.path.join(source_path, filename)
with open(filepath, 'r', encoding='utf-8') as f:
    textblocks = json.load(f)

In [14]:
len(textblocks)

413

In [15]:
textblocks[30][:10]

[{'coordinates': [211.67999267578125,
   1086.2401123046875,
   361.0691833496094,
   1091.0400390625],
  'text': '6. Aeneid.\n',
  'tag': 'margin'},
 {'coordinates': [405.6000061035156,
   200.16000366210938,
   1988.2835693359375,
   217.34410095214844],
  'text': '20\nExamen sententiae Parisiensis scholae\n',
  'tag': 'header'},
 {'coordinates': [573.3599853515625,
   284.6640930175781,
   2086.989501953125,
   289.5841064453125],
  'text': '"Si in mundo sublunari nulla est substantia ab elementaribus quatuor distincta, nulla est quinta essen¬\n',
  'tag': 'header'},
 {'coordinates': [672.47998046875,
   334.7998962402344,
   1029.914794921875,
   339.5998840332031],
  'text': 'tia, quae extrahi possit.\n',
  'tag': 'text'},
 {'coordinates': [573.3599853515625,
   384.9600524902344,
   1760.6229248046875,
   389.7600402832031],
  'text': 'Prius est. Posterius ergo: & per consequens, ignis & opera perditur extrahendo,"\n',
  'tag': 'text'},
 {'coordinates': [504.9599914550781,
   433

In [16]:
# Modify the token extensions for simpler output
if not Token.has_extension("pages"):
    Token.set_extension("pages", default=None)
if not Token.has_extension("textblocks"):
    Token.set_extension("textblocks", default=None)
if not Doc.has_extension("char_to_source"):
    Doc.set_extension("char_to_source", default=None)


def process_textblocks(textblocks):
    full_text = ""
    char_to_source = {}

    for page_idx, page in enumerate(textblocks):
        for tb_idx, tb in enumerate(page):
            if tb["tag"] == "text":
                start_idx = len(full_text)
                text = tomela.text_cleaner(tb["text"])

                for char_idx in range(len(text)):
                    char_to_source[start_idx + char_idx] = {
                        "page_idx": page_idx,
                        "textblock_idx": tb_idx
                    }

                full_text +=  text

    return full_text, char_to_source


@Language.component("source_tracker")
def source_tracker(doc):
    if doc._.char_to_source is not None:
        for token in doc:
            # Get the character span of the entire token
            token_char_range = range(token.idx, token.idx + len(token.text))

            pages = set()
            textblocks = set()

            for char_idx in token_char_range:
                if char_idx in doc._.char_to_source:
                    source_info = doc._.char_to_source[char_idx]
                    pages.add(source_info["page_idx"])
                    textblocks.add(source_info["textblock_idx"])

            token._.pages = sorted(list(pages))
            token._.textblocks = sorted(list(textblocks))
    return doc


# Add the custom component to your existing pipeline if not already added
if "source_tracker" not in tomela.nlp.pipe_names:
    tomela.nlp.add_pipe("source_tracker", before="senter")


def process_with_source_tracking(textblocks, nlp):
    full_text, char_to_source = process_textblocks(textblocks)
    # Create the doc with the text
    doc = nlp.make_doc(full_text)
    # Set the char_to_source before running the pipeline
    doc._.char_to_source = char_to_source
    # Process the doc through each pipeline component
    for name, proc in nlp.pipeline:
        doc = proc(doc)
    return doc

In [17]:
textblocks[21:22]

[[{'coordinates': [584.4000244140625,
    205.20004272460938,
    1917.1199951171875,
    218.6399383544922],
   'text': '11\ncontra Alchymiam latae.\n',
   'tag': 'header'},
  {'coordinates': [151.67999267578125,
    291.1199645996094,
    1958.6729736328125,
    295.9199523925781],
   'text': 'Censore ipso, qui non potest negare, Alchymiam habere bona quaedam, vt aquas stillatitias, & olea, ea¬\n',
   'tag': 'header'},
  {'coordinates': [149.27999877929688,
    339.3599548339844,
    1925.6759033203125,
    344.1599426269531],
   'text': 'que non paruo numero, & non vna forma, ita vt iustam artem haec sola complere possint. Si habet ita¬\n',
   'tag': 'text'},
  {'coordinates': [149.27999877929688,
    387.8640441894531,
    1914.72314453125,
    392.7840576171875],
   'text': 'que bona & benigna coniuncta malignis & deleteriis, tam parùm erat Diabolo asscribenda, quàm pa¬\n',
   'tag': 'text'},
  {'coordinates': [141.83999633789062,
    438.7439270019531,
    1930.4232177734375,
   

In [ ]:
doc = process_with_source_tracking(textblocks, tomela.nlp)

In [19]:
doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)), {"page" : t._.pages, "texblock" : t._.textblocks}) for t in sent]) for sent in doc.sents]
sent_data_updated = []
for n_sent, sent_data in enumerate(doc_sentdata):
    sent_data_updated.append((filename[:6], n_sent, sent_data[0], sent_data[1]))

In [22]:
sent_data_updated[:5]

[('100085',
  0,
  'Andreae Libauii Med.',
  [('Andreae', 'Andreas', 'ADJ', (0, 7), {'page': [1], 'texblock': [3]}),
   ('Libauii', 'Libauii', 'PROPN', (8, 15), {'page': [1], 'texblock': [3]}),
   ('Med', 'ego', 'PROPN', (16, 19), {'page': [1], 'texblock': [3]}),
   ('.', '.', 'PUNCT', (19, 20), {'page': [1], 'texblock': [3]})]),
 ('100085',
  1,
  'D. Sex Libris Declarata.',
  [('D.', 'Decimus', 'PROPN', (0, 2), {'page': [1], 'texblock': [3]}),
   ('Sex', 'Sex', 'NUM', (3, 6), {'page': [1], 'texblock': [5]}),
   ('Libris', 'Liber', 'PROPN', (7, 13), {'page': [1], 'texblock': [5]}),
   ('Declarata', 'declaro', 'PROPN', (14, 23), {'page': [1], 'texblock': [5]}),
   ('.', '.', 'PUNCT', (23, 24), {'page': [1], 'texblock': [5]})]),
 ('100085',
  2,
  'Artis Libro Comprehensarum, AdIectis Fornacum Et Aliorum Uasorum Figuris, partim ex impressis antehac autoribus, partim aliunde acceptis, & ex latibulis officinarum productis.',
  [('Artis', 'ars', 'NOUN', (0, 5), {'page': [1], 'texblock': [8

In [18]:
target_path = "/srv/data/tome/tome-corpus/sents_data_id_jsons_v4-0/"
os.makedirs(target_path, exist_ok=True)

In [19]:
# filename_id_dict.items()

In [20]:
source_path = "../data/emlap_annotated_textblocks/"
len(os.listdir(source_path)) # 100 for textblocks, 100 for parameters used for their extraction

200

In [26]:
os.listdir(source_path)

['100083_Sendivogius1616_Tractatus_de_sulphure_MBS_MDZ_params.json',
 '100084_Croll1609_Basilica_chymica_MDZ_MBS.json',
 '100094_Anon1625_Musaeum_hermeticum_VD17_SLUB.json',
 '100099_Francus1607_De_Arte_Chemica_VD17_Halle_params.json',
 '100044_Dorn1578_Theophrasti_Germani_Principis_MDZ_MBS_params.json',
 '100058_Hagecius1596_Actio_medica_ER_UBB.json',
 '100078_Harvet1605_Demonstratio_veritatis_VD17_FAU_params.json',
 '100013_Ulstad1525_Coelum_philosophorum_Medica_BIUSP.json',
 '100069_Severinus1571_Idea_medicinae_philosophicae_GB_Noscemus.json',
 '100074_Hoghelande1595_De_lapidis_physici_conditionibus_MDZ_MBS.json',
 '100070_Libavius1597_Alchemia_ER_Noscemus_params.json',
 '100066_Suchten1575_De_secretis_antimonii_ONB_params.json',
 '100059_Suavius1567_Theophrasti_Paracelsi_Philosophiae_ONB.json',
 '100053_Mirandola1586_De_auro_libri_tres_MDZ_MBS.json',
 '100072_Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS_params.json',
 '100080_Anon1611_Tratatus_de_secretissimo_MDZ_MBS_params.jso

In [25]:
for filename in os.listdir(source_path):
    if "_params" not in filename:
            id = filename[:6]
            try:
                if id + ".json" not in os.listdir(target_path):
                    # filename = filename.replace(".pdf", ".json")
                    filepath = os.path.join(source_path, filename)
                    with open(filepath, 'r', encoding='utf-8') as f:
                            textblocks_pages = json.load(f)
                    print("currently processing: ", filename)
                    doc = process_with_source_tracking(textblocks_pages, tomela.nlp)
                    doc_sentdata = [(sent.text, [(t.text, t.lemma_, t.pos_, (t.idx - sent[0].idx, t.idx - sent[0].idx + len(t)),  {"page" : t._.pages, "texblock" : t._.textblocks}) for t in sent]) for sent in doc.sents]
                    sent_data_updated = []
                    for n_sent, sent_data in enumerate(doc_sentdata):
                            sent_data_updated.append((id, n_sent, sent_data[0], sent_data[1]))
                    with open(target_path + str(id) + ".json", "w") as f:
                            json.dump(sent_data_updated, f)
            except:
                print("failed with file: ", id, filename)
                pass

currently processing:  100085_Libavius1606_Commentariorum_alchemiae_pars_1_MDZ_MBS.json
currently processing:  100100_Van_Helmont1648_Ortus_medicinae_MDZ_MBS.json
currently processing:  100004_Anon1561_Verae_Alchemiae_MDZ_MBS.json
currently processing:  100072_Andernach1571_De_medicina_veteri_et_novi_MDZ_MBS.json


In [26]:
fns_jsons = os.listdir(target_path)
fns_jsons[:10]

['100044.json',
 '100034.json',
 '100014.json',
 '100094.json',
 '100060.json',
 '100090.json',
 '100010.json',
 '100078.json',
 '100068.json',
 '100079.json']

In [27]:
len(fns_jsons)

100

In [28]:
sents_data = json.load(open(target_path + fns_jsons[20], "r"))
sents_data[100:103]

[['100089',
  100,
  'Ulterius quaerentem sine responso post ualedictionem discedens, me a somno excitatum, in desideratam Eutopiae regionem sistit.',
  [['Ulterius', 'ulterior', 'ADV', [0, 8], {'page': [11], 'texblock': [15]}],
   ['quaerentem', 'quaero', 'VERB', [9, 19], {'page': [11], 'texblock': [15]}],
   ['sine', 'sine', 'ADP', [20, 24], {'page': [11], 'texblock': [15]}],
   ['responso',
    'responsum',
    'NOUN',
    [25, 33],
    {'page': [11], 'texblock': [15]}],
   ['post', 'post', 'ADP', [34, 38], {'page': [11], 'texblock': [15]}],
   ['ualedictionem',
    'ualedictio',
    'NOUN',
    [39, 52],
    {'page': [11], 'texblock': [15, 16]}],
   ['discedens',
    'discedo',
    'VERB',
    [53, 62],
    {'page': [11], 'texblock': [16]}],
   [',', ',', 'PUNCT', [62, 63], {'page': [11], 'texblock': [16]}],
   ['me', 'ego', 'PRON', [64, 66], {'page': [11], 'texblock': [16]}],
   ['a', 'ab', 'ADP', [67, 68], {'page': [11], 'texblock': [16]}],
   ['somno', 'somnus', 'NOUN', [69, 74]

In [37]:
import os, json, pickle
prev_path   = target_path  # where your old .pickle / .json live
target_path = "../data/sents_data_jsons_dicts/"
os.makedirs(target_path, exist_ok=True)

def token_tuple_to_dict(tok):
    """
    Accepts token tuples of length 4 or 5:
      4: (text, lemma, pos, (start, end))
      5: (text, lemma, pos, (start, end), ref_dict)
    Returns a JSON-serializable dict.
    """
    if len(tok) < 4:
        raise ValueError(f"Unexpected token shape: {tok}")

    token_text, lemma, pos, span = tok[0], tok[1], tok[2], tok[3]
    if not isinstance(span, (list, tuple)) or len(span) != 2:
        raise ValueError(f"Bad span in token: {tok}")

    char_start, char_end = int(span[0]), int(span[1])

    # Optional ref at index 4
    ref = tok[4] if len(tok) >= 5 else None

    # Make sure ref is JSON-friendly
    if isinstance(ref, dict):
        page = ref.get("page")
        # convert sets/tuples to lists to be JSON-serializable
        if isinstance(page, (set, tuple)):
            page = list(page)
        ref = {
            "page": page,
            "textblock": ref.get("textblock") or ref.get("texblock")  # tolerate earlier key name
        }
    elif ref is not None:
        # unexpected type → stringify to avoid JSON errors
        ref = str(ref)

    return {
        "token_text": token_text,
        "lemma": lemma,
        "pos": pos,
        "ref": ref,                  # << stays None if we don’t have it
        "char_start": char_start,
        "char_end": char_end,
    }

def sent_tuple_to_dict(entry):
    """
    entry is typically: (work_id, sent_id, sent_text, tokens_list)
    """
    if len(entry) != 4:
        raise ValueError(f"Unexpected sentence shape: {type(entry)} {entry}")

    work_id, sent_id, sent_text, tokens_list = entry
    tokens_dicts = [token_tuple_to_dict(tok) for tok in tokens_list]
    return {
        "work_id": work_id,
        "sent_id": int(sent_id),
        "sent_text": sent_text,
        "tokens_data": tokens_dicts,
    }

def load_any(path):
    """
    Load .pickle OR .json produced by your previous run.
    Must return a list of sentence entries (tuples/lists).
    """
    if path.endswith(".pickle"):
        with open(path, "rb") as f:
            return pickle.load(f)
    elif path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        raise ValueError(f"Unsupported file type: {path}")

for fn in os.listdir(prev_path):
    if not (fn.endswith(".pickle") or fn.endswith(".json")):
        continue

    # Keep your 6-char doc id convention if you like
    doc_id = fn[:6]
    out_path = os.path.join(target_path, f"{doc_id}.json")
    if os.path.exists(out_path):
        continue

    try:
        prev_data = load_any(os.path.join(prev_path, fn))
        # If the previous version stored only (sent_text, tokens) per sentence,
        # reconstruct work_id/sent_id here as needed:
        # e.g., prev_data == [(sent_text, tokens), ...]
        if prev_data and len(prev_data[0]) == 2:
            # synthesize (work_id, sent_id, sent_text, tokens)
            prev_data = [(doc_id, i, s[0], s[1]) for i, s in enumerate(prev_data)]

        sents_dicts = [sent_tuple_to_dict(row) for row in prev_data]

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(sents_dicts, f, ensure_ascii=False, indent=2)

        print("wrote:", out_path)

    except Exception as e:
        print("failed:", fn, "-", e)

wrote: ../data/sents_data_jsons_dicts/100044.json
wrote: ../data/sents_data_jsons_dicts/100034.json
wrote: ../data/sents_data_jsons_dicts/100014.json
wrote: ../data/sents_data_jsons_dicts/100094.json
wrote: ../data/sents_data_jsons_dicts/100060.json
wrote: ../data/sents_data_jsons_dicts/100090.json
wrote: ../data/sents_data_jsons_dicts/100010.json
wrote: ../data/sents_data_jsons_dicts/100078.json
wrote: ../data/sents_data_jsons_dicts/100068.json
wrote: ../data/sents_data_jsons_dicts/100079.json
wrote: ../data/sents_data_jsons_dicts/100043.json
wrote: ../data/sents_data_jsons_dicts/100072.json
wrote: ../data/sents_data_jsons_dicts/100041.json
wrote: ../data/sents_data_jsons_dicts/100012.json
wrote: ../data/sents_data_jsons_dicts/100082.json
wrote: ../data/sents_data_jsons_dicts/100025.json
wrote: ../data/sents_data_jsons_dicts/100019.json
wrote: ../data/sents_data_jsons_dicts/100085.json
wrote: ../data/sents_data_jsons_dicts/100093.json
wrote: ../data/sents_data_jsons_dicts/100095.json


In [32]:
lemmatized_sents_path = "/srv/data/tome/tome-corpus/lemmatized_sents_v4-0/"
try:
    os.mkdir(lemmatized_sents_path)
except:
    pass

In [33]:
os.listdir(lemmatized_sents_path)

[]

In [34]:
for fn in fns_jsons:
    lemmatized_sents = []
    sents_data = json.load(open(target_path + fn, "rb"))
    print(fn)
    for (doc_id, sent_id, sent_text, sent_data) in sents_data:
        lemmasent = []
        for wordform, lemma, tag, position, t_ref in sent_data:
            if tag in ["NOUN", "PROPN", "ADJ", "VERB"]:
                lemmasent.append(lemma.lower())
        lemmatized_sents.append(" ".join(lemmasent) + "\n")
    with open(lemmatized_sents_path + fn.replace(".json", ".txt"), "w", encoding="utf-8") as f:
        f.writelines(lemmatized_sents)

100044.json
100034.json
100014.json
100094.json
100060.json
100090.json
100010.json
100078.json
100068.json
100079.json
100043.json
100072.json
100041.json
100012.json
100082.json
100025.json
100019.json
100085.json
100093.json
100095.json
100089.json
100047.json
100086.json
100013.json
100028.json
100073.json
100070.json
100048.json
100081.json
100088.json
100038.json
100035.json
100042.json
100071.json
100053.json
100083.json
100003.json
100059.json
100002.json
100049.json
100032.json
100052.json
100066.json
100045.json
100096.json
100051.json
100001.json
100054.json
100065.json
100064.json
100004.json
100076.json
100092.json
100050.json
100046.json
100069.json
100058.json
100062.json
100099.json
100087.json
100100.json
100033.json
100006.json
100029.json
100017.json
100061.json
100007.json
100074.json
100027.json
100063.json
100037.json
100067.json
100026.json
100097.json
100023.json
100056.json
100024.json
100091.json
100040.json
100021.json
100055.json
100005.json
100080.json
1000

In [36]:
import shutil
shutil.copytree("/srv/data/tome/tome-corpus/lemmatized_sents_v4-0/", "../data/lemmatized_sents", dirs_exist_ok=True)
shutil.copytree("/srv/data/tome/tome-corpus/sents_data_id_jsons_v4-0/", "../data/sents_data", dirs_exist_ok=True)

'../data/sents_data'